# Absorbing objects

Walls and `simgeometry` shapes that remove the particles reaching them, with every removal recorded per wall and object: the currents to each electrode, and steps tested along their whole length so that no particle jumps over a thin plate.

A beam of like charges leaves a source on the left. It meets a slit electrode (an `Aperture2D`, two plates 0.02 thick), then a round obstacle, inside a round chamber (an inverted `Circle`, which absorbs everything outside it), and ends at a collector wall at $x = 1$. Units are arbitrary here ($k_e = 1$). `absorbing_walls` takes axis-aligned walls by name; `absorbing_objects` takes any 2D shape under a name of your choice.

In [ ]:
import numpy as np
import plasma_plots
from simgeometry import Aperture2D, Circle

from tree_d_3d import ChargedBody, ElectrostaticSimulation, Quad

plasma_plots.set_backend("plotly")

rng = np.random.default_rng(1)
n = 400
bodies = [
    ChargedBody(x, y, vx=vx, vy=vy, charge=1e-4, mass=1.0)
    for x, y, vx, vy in zip(
        rng.uniform(-0.95, -0.7, n),
        rng.normal(0.0, 0.25, n),
        rng.normal(1.0, 0.05, n),
        rng.normal(0.0, 0.08, n),
    )
]
objects = {
    "slit": Aperture2D.slit(-0.3, 0.12, 0.02, 0.8),
    "obstacle": Circle(midpoint=(0.35, 0.05), radius=0.08),
    "chamber": Circle(midpoint=(0.0, 0.0), radius=1.3, inverted=True),
}
sim = ElectrostaticSimulation(
    bodies,
    Quad(0.0, 0.0, 1.5),
    k_e=1.0,
    softening=1e-3,
    dt=0.01,
    absorbing_walls={"x_max": 1.0},
    absorbing_objects=objects,
)
for _ in range(160):
    sim.step()
print(f"{sim.num_particles} particles left after t = {sim.current_time:.2f}")
print("absorbed:", sim.absorbed.counts())

`sim.absorbed` holds every removed particle: its position just after it crossed the boundary, its velocity, mass, charge, the time and the name of the wall or object. The charge per name divided by the run time is the mean current to that electrode.

In [ ]:
for name, charge in sim.absorbed.charges().items():
    print(
        f"{name:9s} charge {charge:.2e}, mean current {charge / sim.current_time:.2e}"
    )
print(
    f"absorbed + flying: {sim.absorbed.total_charge + sim.get_total_charge():.2e}, as at the start"
)
print(f"kinetic energy carried out: {sim.absorbed.kinetic_energy:.1f}")

with plasma_plots.figure(1, 1) as fig:
    ax = fig[0]
    for shape in objects.values():
        for loop in shape.outlines(0.01):
            ax.plot(*np.concatenate((loop, loop[:, :1]), axis=1), color="gray")
    for name in sim.absorbed.counts():
        hit = sim.absorbed.wall == name
        ax.plot(*sim.absorbed.position[:, hit], ".", label=name)
    ax.plot(*sim.positions, ".", color="black", label="still flying")
    ax.axvline(1.0, color="gray", linestyle="--")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
    ax.legend()
fig

The whole step is tested, not only where it ends. Here particles move 0.2 per step, ten times the plate thickness. A test of the end points alone would miss every one of them, since they end beyond the plate. The swept test (simgeometry's compiled polygon kernel for `Polygon` and `Aperture2D`, `crossings` for other shapes) catches them all.

In [ ]:
slit = Aperture2D.slit(0.0, 0.2, 0.02, 0.6)
heights = np.linspace(-0.75, 0.75, 31)
fast = [ChargedBody(-0.3, y, vx=1.0, charge=0.0) for y in heights]
sim = ElectrostaticSimulation(
    fast, Quad(0.0, 0.0, 1.0), k_e=1.0, dt=0.2, absorbing_objects={"slit": slit}
)
for _ in range(3):
    sim.step()
ends = np.stack((np.full_like(heights, 0.3), heights))
print(f"swept test: {sim.absorbed.counts().get('slit', 0)} of {heights.size} absorbed")
print(f"end points alone: {int(np.sum(slit.contains_points(ends)))} inside the plates")
print(f"through the aperture: {sim.num_particles}")